# 2 · Conjuring geometry & taming the mesh ⚔️🛡️

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [⌛ ~1 min](/lite/notebooks/index.html?path=02-geometry.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/02-geometry.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time). Use the ⚙ **View options** to
switch story / quizzes / gimmicks on or off.</sub>
:::

:::{dropdown} 🎭 Story — to the forge
:class: storytelling

*Before any adventurer rides out, they visit the **forge**. The Beast you have met; now
you must arm yourself. You will **hammer out a sword and a shield** — not from steel, but
from pure geometry — and learn, in the doing, how to **sketch shapes** and **tame the mesh**
that NGSolve lays over them.*
:::

Unit 1 began in **3D** (the Beast). Here we learn to **make geometry ourselves**: **2D**
sketches (a sword and a shield), the **knobs** that control a mesh, **named regions**
(materials & boundaries) on the sword & shield in **3D**, and **1D** meshes. Three short
supplements go further — querying a mesh's **topology**, an **adaptive refinement** step,
and **importing a real external model**.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
# --- Bring the website's UI into this live notebook: the ⚙ View-options panel,
# the foldable story/how-to/quiz/further-reading categories and the gimmicks
# (rolling logo + winking head). Loads static/custom.css + view-options.js via
# notebooks/data/ngsum_ui.py. A no-op on the static-site build. --------------
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):
    sys.path.insert(0, os.path.join(os.getcwd(), "data"))
    try:
        import ngsum_ui; ngsum_ui.enable()
    except Exception:
        pass

In [ ]:
from netgen.occ import WorkPlane, OCCGeometry, Axis, Axes, Pnt, X, Y, Z, Glue
from netgen.meshing import MeshingParameters
from ngsolve import Mesh, H1, GridFunction, x, y, VOL, BND, Integrate, CF
from ngsolve.meshes import Make1DMesh
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
from math import sqrt

## 1. Forge a sword — a 2D sketch

The 2D workflow is **sketch → face**. A `WorkPlane` is a pen on a sheet of paper:
`MoveTo` puts it down, each `LineTo` draws a straight segment, `Close` joins back to the
start, and `.Face()` fills the closed outline. We trace the **silhouette of a sword** in one
loop — blade, crossguard, grip, pommel. Here is the exact path first, so you can read the
code below as a walk from point to point:

```{image} data/sword-sketch.png
:alt: The sword outline as 14 numbered points traced counter-clockwise into one closed loop
:width: 320px
:align: center
```

:::{important}
**Orientation matters.** A face's outline must run **counter-clockwise** (so it encloses a
*positive* area). Trace it clockwise and the area comes out negative — the mesher then
silently fails to triangulate it. We walk **tip → left → pommel → right → back** (points
0 → 13 in the sketch).
:::

In [ ]:
def sword_2d():
    """Silhouette of a sword as one counter-clockwise outline (the 14 points of the sketch)."""
    return (WorkPlane()
            .MoveTo(0, 10)                                            # 0: the tip
            .LineTo(-0.5, 2.2).LineTo(-2.2, 2.2).LineTo(-2.2, 1.5)    # 1–3: left blade edge → guard
            .LineTo(-0.35, 1.5).LineTo(-0.35, -1.6)                   # 4–5: into the grip, down
            .LineTo(-0.95, -2.2).LineTo(0, -2.95).LineTo(0.95, -2.2)  # 6–8: the diamond pommel
            .LineTo(0.35, -1.6).LineTo(0.35, 1.5)                     # 9–10: back up the grip
            .LineTo(2.2, 1.5).LineTo(2.2, 2.2).LineTo(0.5, 2.2)       # 11–13: guard → right blade edge
            .Close().Face())

sword = sword_2d()
print(f"the sword: area {sword.mass:.1f}  (positive → correctly oriented)")
mesh_sw = Mesh(OCCGeometry(sword, dim=2).GenerateMesh(maxh=0.4))
print(f"meshed: {mesh_sw.ne} triangles")
Draw(mesh_sw)

## 2. A shield — curved edges and the meshing knobs

A straight-line sketch is easy; a **curved** boundary is where meshing gets interesting.
`Spline` draws a smooth curve through points — we use it for the rounded bottom of a
**heater shield**. Two knobs control the mesh:

* **`GenerateMesh(maxh=…)`** — the global maximum element size. Smaller ⇒ more, smaller
  triangles ⇒ more accurate but more expensive.
* **`mesh.Curve(order)`** — straight-sided elements approximate a curve by a crude
  **polygon**; `Curve` *bends* the element edges to follow the true boundary.

In [ ]:
def shield_2d(W=3.0, H=3.5):
    """A heater shield: flat top, sides splining down to a point."""
    return (WorkPlane()
            .MoveTo(-W, H).LineTo(-W, 0.3)                            # top-left, down left side
            .Spline([(-W*0.6, -2.0), (0, -H)])                       # curve to the bottom point
            .Spline([(W*0.6, -2.0), (W, 0.3)])                       # curve up the right side
            .LineTo(W, H).Close().Face())                            # up right, close the top

shield = shield_2d()
for h in (0.8, 0.35):
    m = Mesh(OCCGeometry(shield, dim=2).GenerateMesh(maxh=h))
    print(f"maxh={h}:  {m.ne:4d} triangles")

mesh_sh = Mesh(OCCGeometry(shield, dim=2).GenerateMesh(maxh=0.35))
mesh_sh.Curve(3)                                                     # bend elements onto the curve
Draw(mesh_sh)

### More meshing knobs — `MeshingParameters`

`maxh` and `Curve` are the two you reach for most, but Netgen exposes many more through a
**`MeshingParameters`** object (or directly as keyword arguments to `GenerateMesh`). A few
worth knowing:

* **`grading`** (0–1) — how fast the element size may change from one element to the next.
  Small (≈0.1) ⇒ smooth, gradual size changes (more elements); large (≈0.9) ⇒ abrupt.
* **`segmentsperedge`** — the minimum number of boundary segments per geometry edge; raise
  it to resolve small edges.
* **`optsteps2d` / `optsteps3d`** — how many mesh-optimisation sweeps to run (better element
  shapes, at a little cost).
* **`delaunay`** — use the Delaunay algorithm for the fill (on by default).
* **`quad_dominated`** — prefer quadrilaterals over triangles on surfaces.

(`MeshingParameters?` prints the full list.) `grading` is clearest next to a **locally
refined feature**: we refine the sword's tip edge, then vary only the grading — a small
grading lets the fine zone **spread out** (many more triangles), a large one keeps it local:

In [ ]:
for g in (0.7, 0.2):
    sw = sword_2d()
    sw.edges.Max(Y).maxh = 0.1                                       # a fine feature at the tip
    mp = MeshingParameters(maxh=0.7, grading=g)                      # only the grading differs
    m = Mesh(OCCGeometry(sw, dim=2).GenerateMesh(mp=mp))
    note = "gradual → fine zone spreads" if g < 0.5 else "abrupt → fine zone stays local"
    print(f"grading={g}:  {m.ne:4d} triangles  ({note})")

# the same options can also be passed straight as keyword arguments to GenerateMesh:
m_kw = Mesh(OCCGeometry(shield, dim=2).GenerateMesh(maxh=0.6, grading=0.2, optsteps2d=5))
Draw(m_kw)

**Local refinement.** You rarely want the *whole* mesh fine. Setting **`.maxh` on a single
sub-shape** refines only there — here the **sharp tip** of the blade, where the geometry is
thin and the solution will vary fastest, while the broad grip stays coarse.

In [ ]:
sword_ref = sword_2d()
sword_ref.edges.Max(Y).maxh = 0.12                                  # fine only near the tip edges
mesh_ref = Mesh(OCCGeometry(sword_ref, dim=2).GenerateMesh(maxh=0.6))
print(f"locally refined sword: {mesh_ref.ne} triangles (fine tip, coarse grip)")
Draw(mesh_ref)

## 3. Named regions — materials & boundaries, in 3D

Real models are made of **named parts**. A 2D face becomes a **3D solid** by **extrusion**
(`face.Extrude(d)` pulls it a depth `d` along its normal); we give the sword and shield a
thickness and arrange them into a little **scene**. Crucially, we **name** them: a solid's
`.name` is its **material** (a *volume* region), and `.faces.name` names its surface as a
**boundary** region. These names are exactly how every later unit attaches **material
coefficients** and **boundary conditions**.

In [ ]:
sword3d  = sword_2d().Extrude(0.5)
shield3d = shield_2d().Extrude(1.0).Move((0, 0, -3))                 # shield set behind the blade
sword3d  = sword3d.Rotate(Axis(Pnt(0, 0, 0), Z), 25).Move((1.5, -1, -6))

sword3d.name  = "sword"                                             # material name (a volume region)
shield3d.name = "shield"
sword3d.faces.name  = "blade_surf"                                 # boundary name (all its faces)
shield3d.faces.name = "shield_surf"

scene = Glue([sword3d, shield3d])
mesh3d = Mesh(OCCGeometry(scene).GenerateMesh(maxh=1.2)); mesh3d.Curve(2)
print(f"the 3D loadout: {mesh3d.ne} tetrahedra")
Draw(mesh3d)

**Inspecting regions.** Ask the mesh what it carries: `GetMaterials()` / `GetBoundaries()`
list the names, while `mesh.Materials(pattern)` / `mesh.Boundaries(pattern)` select a
**`Region`** (a mask over elements) by a **regex** pattern — `"sword"`, `"sword|shield"`,
`".*"`. Each element also knows its own material as `el.mat`.

In [ ]:
print("materials :", mesh3d.GetMaterials())                        # ('sword', 'shield')
print("boundaries:", sorted(set(mesh3d.GetBoundaries())))          # ['blade_surf', 'shield_surf']

# each element knows its own material, so we can count per sub-domain:
n_sword  = sum(1 for el in mesh3d.Elements(VOL) if el.mat == "sword")
n_shield = sum(1 for el in mesh3d.Elements(VOL) if el.mat == "shield")
print(f"tetrahedra — sword: {n_sword}, shield: {n_shield}, total: {mesh3d.ne}")

# mesh.Materials(pattern) returns a Region you can integrate over or restrict a space to:
vol_sword = Integrate(CF(1), mesh3d, definedon=mesh3d.Materials("sword"))
print(f"volume of the 'sword' region: {vol_sword:.2f}")

## 4. Down a dimension — 1D meshes

A mesh can be **one-dimensional** too — a chain of intervals on $[0,1]$. `Make1DMesh(n)`
spaces $n$ of them **uniformly**; a `mapping` grades them, packing points where you need
resolution (say near a boundary layer).

In [ ]:
def nodes(m): return sorted(p[0] for p in m.ngmesh.Points())
uniform = Make1DMesh(12)
graded  = Make1DMesh(12, mapping=lambda t: t**1.7)                  # clustered near x=0
fig, ax = plt.subplots(figsize=(7, 1.4))
ax.plot(nodes(uniform), [1]*13, "o-", label="uniform")
ax.plot(nodes(graded),  [0]*13, "o-", label="graded $t^{1.7}$")
ax.set_yticks([0, 1]); ax.set_yticklabels(["graded", "uniform"]); ax.set_xlabel("x")
ax.legend(loc="center right"); ax.set_ylim(-0.5, 1.5); fig.tight_layout()

---
## Supplementary A — knowing your mesh: topology, iterators & a value at a point

*(Supplementary.)* A mesh is not just a picture: you can **walk its topology** and **evaluate
functions on it**. `mesh.nv / mesh.ne` count vertices and elements; you can **iterate** over
elements (`mesh.Elements(VOL)`), each of which knows its `vertices` and material; and
`mesh(x, y)` locates the element containing a point so a `CoefficientFunction` can be
evaluated there (the bridge to unit 3). The i-tutorial
[1.8 Exploring the mesh topology](https://docu.ngsolve.org/latest/i-tutorials/unit-1.8-meshtopology/meshtopology.html)
goes the whole way — edges, faces, and the element→dof maps.

In [ ]:
print(f"the sword mesh has {mesh_sw.nv} vertices and {mesh_sw.ne} triangles")

# walk the elements: each one knows its vertices (by number) and its material
for el in list(mesh_sw.Elements(VOL))[:3]:
    print(f"  triangle {el.nr}: vertices {[v.nr for v in el.vertices]}, material '{el.mat}'")
print(f"  … and {mesh_sw.ne - 3} more")

# evaluate a function at world points: mesh(x, y) finds the containing element
gf = GridFunction(H1(mesh_sw, order=2))                             # a function living on the mesh
gf.Set(x*x + y*y)                                                   # = squared distance from the hilt
for px, py in [(0, 8), (0, 0), (1.5, 1.7)]:                         # tip, centre, guard
    print(f"  at ({px:>4},{py:>4}):  x²+y² = {gf(mesh_sw(px, py)):6.2f}")
Draw(gf, mesh_sw, "x²+y²")

---
## Supplementary B — an adaptive refinement step

*(Supplementary.)* Instead of refining everywhere, we can **mark** individual elements and
refine only those. A real adaptive loop marks by an **error estimator** (unit 10); here we
use a toy criterion — **proximity to a chosen world point** — to show the mechanics. We set
NGSolve's per-element **refinement flag** with `mesh.SetRefinementFlag(el, …)`, then call
**`mesh.Refine()`**, which bisects the marked elements (plus a small conforming closure).
See the i-tutorial
[1.6 Adaptivity](https://docu.ngsolve.org/latest/i-tutorials/unit-1.6-adaptivity/adaptivity.html).

In [ ]:
mesh_ad = Mesh(OCCGeometry(shield, dim=2).GenerateMesh(maxh=0.6))
focus = (0.0, -3.0)                                                 # refine near the shield's tip
for step in range(3):                                              # three adaptive sweeps
    for el in mesh_ad.Elements(VOL):
        cx, cy = (sum(c) / 3 for c in zip(*[mesh_ad[v].point for v in el.vertices]))
        mesh_ad.SetRefinementFlag(el, sqrt((cx - focus[0])**2 + (cy - focus[1])**2) < 1.0)
    mesh_ad.Refine()
print(f"after 3 marked refinements near {focus}: {mesh_ad.ne} triangles (fine only there)")
Draw(mesh_ad)

**`mesh.Refine()` vs `mesh.ngmesh.Refine()`.** Two very different refinements:

* **`mesh.Refine()`** (NGSolve) is **marked & adaptive** — it bisects the elements you flagged
  (plus a conforming closure) and keeps the **NGSolve mesh and its FE-space hierarchy in
  sync**, so a `GridFunction` can be **prolongated** onto the finer mesh.
* **`mesh.ngmesh.Refine()`** refines the **underlying Netgen mesh uniformly** — every element
  is split and the flags are ignored — bypassing NGSolve's refinement bookkeeping.

Side by side, from the same coarse mesh:

In [ ]:
def coarse(): return Mesh(OCCGeometry(shield, dim=2).GenerateMesh(maxh=0.6))

ma = coarse(); n0 = ma.ne
for el in ma.Elements(VOL):                                        # mark just the bottom strip
    cy = sum(ma[v].point[1] for v in el.vertices) / 3
    ma.SetRefinementFlag(el, cy < -2.0)
ma.Refine()

mu = coarse(); mu.ngmesh.Refine()                                  # uniform: ignores any marks
print(f"start: {n0} triangles")
print(f"  mesh.Refine()        (marked)  -> {ma.ne:4d}  — only the bottom got finer")
print(f"  mesh.ngmesh.Refine() (uniform) -> {mu.ne:4d}  — everything split")

:::{dropdown} 🧠 Quiz — `mesh.Refine()` vs `mesh.ngmesh.Refine()`?
:class: quiz
**`mesh.ngmesh.Refine()`** refines the **Netgen** mesh **uniformly** — every element is split,
the refinement flags are ignored — and it works at the raw geometry/mesh level only.
**`mesh.Refine()`** is NGSolve's **adaptive, marked** refinement: it bisects the elements you
flagged with `SetRefinementFlag` (plus a conforming closure) **and** updates NGSolve's
refinement hierarchy, so finite-element spaces and `GridFunction`s on the mesh stay valid and
can be prolongated. Rule of thumb: **adaptive / FE-aware → `mesh.Refine()`; a quick uniform
split of the raw mesh → `mesh.ngmesh.Refine()`**.
:::

---
## Supplementary C — importing a real external model

*(Supplementary.)* Geometry need not be sketched by hand — NGSolve reads common CAD/mesh
formats. Here we **import an `.stl` model** (a surface triangulation — a blocky *Minecraft
sword*) and let Netgen build a **volume mesh** from it. The same `OCCGeometry` route reads
**STEP/IGES/BREP** CAD files, and imported parts can be **combined** with sketched ones via
the boolean operators (`+`, `-`, `*`, `Glue`) you have already met.

In [ ]:
from netgen.stl import STLGeometry
imported = STLGeometry("data/minecraft-sword.stl")
mesh_mc = Mesh(imported.GenerateMesh(maxh=12))                      # coarse: the model is detailed
print(f"imported Minecraft sword: {mesh_mc.ne} tetrahedra, {mesh_mc.nv} vertices")
Draw(mesh_mc)

:::{dropdown} 📚 Further reading
:class: further-reading

- **Netgen/OCC geometry** — the i-tutorials on
  [OpenCASCADE geometry](https://docu.ngsolve.org/latest/i-tutorials/unit-4.4-occ/occ.html),
  [workplanes](https://docu.ngsolve.org/latest/i-tutorials/unit-4.4-occ/workplane.html) and
  [manual meshing](https://docu.ngsolve.org/latest/i-tutorials/unit-4.3-manualmesh/manualmeshing.html).
- **Named regions — materials & boundaries** — i-tutorial
  [1.5 Spaces and forms on sub-domains](https://docu.ngsolve.org/latest/i-tutorials/unit-1.5-subdomains/subdomains.html).
- **Mesh topology & iterators** — i-tutorial
  [1.8 Exploring the mesh topology](https://docu.ngsolve.org/latest/i-tutorials/unit-1.8-meshtopology/meshtopology.html).
- **Adaptivity** — i-tutorial
  [1.6 Adaptivity](https://docu.ngsolve.org/latest/i-tutorials/unit-1.6-adaptivity/adaptivity.html).
- **A guided OCC walkthrough** — the ngs24 tutorial
  [OpenCASCADE Technology geometry](https://docu.ngsolve.org/ngs24/tutorials/03_occ.html).
:::

:::{dropdown} 🧠 Quiz — Netgen `Mesh` vs NGSolve `Mesh`?
:class: quiz
Two different objects with the same nickname. **`netgen.meshing.Mesh`** (you reach it as
`mesh.ngmesh`) is **Netgen's** mesh: the raw geometric/topological data — points, volume and
surface elements, boundary descriptors — produced by `GenerateMesh()`. **`ngsolve.comp.Mesh`**
**wraps** that for finite elements: it adds the FEM view — **named regions** (`Materials`,
`Boundaries`), integration, point evaluation `mesh(x, y)`, mesh deformation, and the
refinement hierarchy — and is what every FE space is built on. `Mesh(ngmesh)` wraps a Netgen
mesh into an NGSolve one; `mesh.ngmesh` gets the Netgen mesh back out.
:::

:::{dropdown} 🧠 Quiz — two ways to build the *same* 2D geometry
:class: quiz
A `WorkPlane` face can be meshed **as a 2D domain** or **as a 2D surface sitting in 3D** — the
same rectangle, two interpretations:

```python
from netgen.occ import WorkPlane, OCCGeometry, Axes, Z

# (1) a genuine 2D geometry — triangles fill the area (mesh.dim == 2)
face   = WorkPlane().Rectangle(2, 1).Face()
mesh2d = Mesh(OCCGeometry(face, dim=2).GenerateMesh(maxh=0.3))

# (2) the same face placed in 3D space — meshed as a SURFACE (mesh.dim == 3, only BND elements)
face3d = WorkPlane(Axes((0, 0, 0), n=Z)).Rectangle(2, 1).Face()
surf   = Mesh(OCCGeometry(face3d).GenerateMesh(maxh=0.3))
```

Both describe the identical 2-by-1 rectangle. Version (1) is a flat **2D domain** (volume
elements, `mesh.dim == 2`); version (2) is a **surface mesh embedded in 3D** (`mesh.dim == 3`,
only boundary elements) — the natural setting for PDEs *on surfaces*, which we solve on the
Beast's curved skin in unit 15.
:::

**Armed and ready.** You can sketch in 2D, tune the meshing knobs, refine where it matters,
name and inspect regions in 3D, and drop to 1D — plus query a mesh, refine it adaptively, and
import a model. Next we meet the one object NGSolve evaluates *everywhere* — the
**CoefficientFunction**.

In [ ]:
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):
    _nb, _title = "03-coefficientfunctions", "3 · What is a CoefficientFunction? 🔨"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next unit:** [" + _title + "](" + _u + ")"))